# SH01: zoom, captions, branding and image template test (Kaggle)

Run **top to bottom in a fresh Kaggle session**. This notebook starts your SH01 FastAPI server on **127.0.0.1:5000** and uses its HTTP endpoints for the image template and video edit. The only local media processing is preparing the already-downloaded MKV if a synced MP4 is unavailable. `ffprobe` checks the inputs and results; Python handles API requests and files.

Add Kaggle dataset `vivanjaiswal/yt-window-local` (the MKV at `/kaggle/input/datasets/vivanjaiswal/yt-window-local/youtube-window-local.mkv`). If you already attached a synced MP4, set `KAGGLE_SYNCED_DATASET_MP4` to its absolute path before cell 3. The initial source window defaults to **5–65 seconds** of the synced three-minute clip for a shorter test; change the markers to **0–180** to render the full clip.

Set Kaggle secrets: `MEDIA_CLIP_API_KEY`, `MEDIA_JOB_STORE_API_KEY`, `BACKBLAZE_KEY_ID`, `BACKBLAZE_APPLICATION_KEY`, `BACKBLAZE_BUCKET_NAME`. Never paste their values into a notebook. The test uses a **shared Cloudflare queue**; ensure this Kaggle session is its sole opted-in high-resource consumer while processing a Kaggle-local `source_path`. Do not run a second high-resource consumer with access to the same queue: it cannot read this session's file.

The four sample logos below are public, generic GitHub topic icons. Replace their URLs with your own public PNG/WebP logos when ready. The image-template background also uses a sample icon unless you set `IMAGE_BACKGROUND_URL` to a public image. Kaggle must remain running until the job completes.


## 1. Pull the latest SH01 revision and install its dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/kaggle/working/sh01-fastapi-server')
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO, check=True)
elif REPO.exists():
    raise RuntimeError(f'Inspect the non-Git directory before proceeding: {REPO}')
else:
    subprocess.run(['git', 'clone', 'https://github.com/BarrieBretton/sh01-fastapi-server.git', str(REPO)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                '-r', str(REPO / 'requirements.txt')], check=True)
revision = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO, text=True).strip()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('SH01 revision:', revision)


## 2. Load private settings and reject older code before starting a long render

In [ ]:
%%bash

cat > .env <<'EOF'

... # REDACTED # ...

EOF

### sanity check for installed pillow version (required for kaggle)

In [ ]:
%%bash
python -m pip install --force-reinstall --no-cache-dir Pillow==12.1.0
python -c "import PIL; from PIL import Image, ImageDraw, ImageFont; print('Pillow OK:', PIL.__version__)"

In [ ]:
import inspect
from dotenv import load_dotenv

load_dotenv(REPO / '.env', override=False)
required = ('MEDIA_CLIP_API_KEY', 'MEDIA_JOB_STORE_API_KEY', 'BACKBLAZE_KEY_ID',
            'BACKBLAZE_APPLICATION_KEY', 'BACKBLAZE_BUCKET_NAME')
try:
    from kaggle_secrets import UserSecretsClient
    secrets_client = UserSecretsClient()
except ImportError:
    secrets_client = None
for name in required:
    if not os.environ.get(name) and secrets_client is not None:
        try:
            value = secrets_client.get_secret(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError('Set these Kaggle secrets: ' + ', '.join(missing))

INPUT_DIR = REPO / 'downloads' / 'caption-input'
INPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.update({
    'CONTROL_PLANE_BASE_URL': 'https://infra-control.vivojaymail.workers.dev',
    'MEDIA_LOCAL_SOURCE_DIR': str(INPUT_DIR),
    'MEDIA_JOB_STORE_MODE': 'remote',
    'MEDIA_QUEUE_CONSUMER_ENABLED': 'true',
    'MEDIA_ALLOW_UNLIMITED_RESOURCES': 'true',
    'MEDIA_CLIP_WHISPER_VAD': 'true',
    'MEDIA_CLIP_WHISPER_MODEL': 'small',
    'MEDIA_CLIP_RENDER_WIDTH': '1080',
    'MEDIA_CLIP_RENDER_HEIGHT': '1920',
    'AUTO_EDIT_MAX_SEGMENTS': '20',
})

from auto_editor import AutoEditRequest, _render_variant
from media_branding import Branding, animated_ass
from media_job_store import store
from image_templating import RenderTemplateRequest

expected_fields = {'zoom_shots', 'zoom_overrides', 'zoom_amount', 'zoom_ramp_seconds',
                   'branding', 'captions_enabled', 'start_seconds', 'end_seconds', 'crop_x_px'}
assert expected_fields <= set(AutoEditRequest.model_fields)
assert {'branding', 'style_preset', 'text_color', 'text_font_size_px'} <= set(RenderTemplateRequest.model_fields)
assert hasattr(store, 'queue_claim'), 'Old queue code; pull latest main before starting the server'
if 'for shot_idx, (seg, out_start) in enumerate(' not in inspect.getsource(_render_variant):
    raise RuntimeError('Missing final edit-decision-list zoom fix in auto_editor.py; replace and deploy it first')
if 'reads_with_travel' not in inspect.getsource(animated_ass):
    raise RuntimeError('Missing corrected marquee reading order in media_branding.py')
print('Updated server fields and both rendering fixes are present. Secrets are not displayed.')


## 3. Reuse the synced MP4, or normalize the attached MKV once

Only this preparation cell uses local FFmpeg. The subsequent captioning, cuts, effects, gradients, logos, edge text, and final render all use SH01 HTTP endpoints. A normalizing re-encode can take a few minutes; it is skipped on subsequent runs if the synced MP4 already exists and passes checks.

In [ ]:
import json
import shutil

DATASET_MKV = Path('/kaggle/input/datasets/vivanjaiswal/yt-window-local/youtube-window-local.mkv')
SYNCED_DATASET = Path(os.environ.get('KAGGLE_SYNCED_DATASET_MP4',
    '/kaggle/input/datasets/vivanjaiswal/yt-window-synced/youtube-window-synced.mp4'))
SYNCED = INPUT_DIR / 'youtube-window-synced.mp4'

def probe(path):
    return json.loads(subprocess.check_output([
        'ffprobe', '-v', 'error',
        '-show_entries', 'stream=codec_type,codec_name,width,height,start_time,duration',
        '-show_entries', 'format=duration', '-of', 'json', str(path),
    ], text=True))

def valid_synced(path):
    if not path.is_file() or not path.stat().st_size:
        return False
    try:
        result = probe(path)
        v = next(s for s in result['streams'] if s['codec_type'] == 'video')
        a = next(s for s in result['streams'] if s['codec_type'] == 'audio')
        return (v['width'], v['height']) == (1920, 1080) and all(
            abs(float(s.get('start_time', 0))) < 0.08 for s in (v, a)
        ) and 179 <= float(result['format']['duration']) <= 180.5
    except (OSError, KeyError, ValueError, StopIteration, subprocess.CalledProcessError):
        return False

if valid_synced(SYNCED):
    print('Reusing:', SYNCED)
elif valid_synced(SYNCED_DATASET):
    shutil.copyfile(SYNCED_DATASET, SYNCED)
    print('Copied synced MP4 from Kaggle input:', SYNCED_DATASET)
else:
    if not DATASET_MKV.is_file():
        raise FileNotFoundError(f'Attach {DATASET_MKV}, or provide a synced MP4 in {SYNCED}')
    mkv = INPUT_DIR / 'youtube-window-local.mkv'
    if not mkv.is_file() or mkv.stat().st_size != DATASET_MKV.stat().st_size:
        shutil.copyfile(DATASET_MKV, mkv)
    tracks = probe(mkv)['streams']
    v = next(s for s in tracks if s['codec_type'] == 'video')
    a = next(s for s in tracks if s['codec_type'] == 'audio')
    delay = float(a.get('start_time', 0)) - float(v.get('start_time', 0))
    if not 0 <= delay <= 5:
        raise RuntimeError(f'Unexpected source A/V offset {delay:.3f}s; inspect the MKV')
    print(f'Normalizing {delay:.3f}s A/V offset; this step can take a few minutes...')
    subprocess.run([
        'ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'warning', '-stats', '-y',
        '-ss', f'{delay:.3f}', '-i', str(mkv), '-t', '180',
        '-map', '0:v:0', '-map', '0:a:0', '-vf', 'setpts=PTS-STARTPTS',
        '-af', 'aresample=async=1:first_pts=0,asetpts=PTS-STARTPTS',
        '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '18', '-pix_fmt', 'yuv420p',
        '-c:a', 'aac', '-b:a', '192k', '-ar', '48000', '-movflags', '+faststart', str(SYNCED),
    ], check=True)
if not valid_synced(SYNCED):
    raise RuntimeError('Synced video failed stream validation: ' + str(probe(SYNCED)))
print('Input:', SYNCED, '| size:', round(SYNCED.stat().st_size / 1048576, 1), 'MiB')


## 4. Edit the test configuration

Change source markers, crop position, logo URLs, colors, marquee strings, and zoom shot numbers here. Shot numbers are based on **selected output shots after silence cutting**, starting at 1. A shot-specific override also applies if that shot is outside `zoom_shots`. The example turns zoom **off** on shot 3, varies amount/ramp on shot 2, and leaves the normal and highlighted caption colors distinct. This config exercises four logos, two main gradients, two side gradients, static top text, a connected left→top→right path, and two single-edge marquees.

In [ ]:
import uuid

# A quick preview uses a 60-second slice. For the full file, set 0, 180, 180.
SOURCE_START_SECONDS = 5.0
SOURCE_END_SECONDS = 65.0
TARGET_SECONDS = 50.0

# Generic PUBLIC PNG URLs; substitute your own four accessible brand/logo URLs.
LOGO_URLS = {
    'top_left': 'https://raw.githubusercontent.com/github/explore/main/topics/python/python.png',
    'top_right': 'https://raw.githubusercontent.com/github/explore/main/topics/javascript/javascript.png',
    'bottom_left': 'https://raw.githubusercontent.com/github/explore/main/topics/docker/docker.png',
    'bottom_right': 'https://raw.githubusercontent.com/github/explore/main/topics/github/github.png',
}
IMAGE_BACKGROUND_URL = os.environ.get('IMAGE_BACKGROUND_URL', LOGO_URLS['top_left'])

BRANDING = {
    'gradients': [
        {'edge': 'top', 'thickness_px': 160,
         'outer': {'color': '#42225F', 'opacity': .78},
         'inner': {'color': '#C66B7B', 'opacity': 0}},
        {'edge': 'bottom', 'thickness_px': 220,
         'outer': {'color': '#061C34', 'opacity': .92},
         'inner': {'color': '#F28642', 'opacity': .08}},
        {'edge': 'left', 'thickness_px': 80,
         'outer': {'color': '#0FC8D8', 'opacity': .17},
         'inner': {'color': '#0FC8D8', 'opacity': 0}},
        {'edge': 'right', 'thickness_px': 80,
         'outer': {'color': '#F15D71', 'opacity': .17},
         'inner': {'color': '#F15D71', 'opacity': 0}},
    ],
    'logos': [
        {'url': LOGO_URLS['top_left'], 'corner': 'top_left', 'size_px': 48,
         'opacity': .85, 'margin_px': 34},
        {'url': LOGO_URLS['top_right'], 'corner': 'top_right', 'size_px': 56,
         'opacity': .8, 'margin_px': 34},
        {'url': LOGO_URLS['bottom_left'], 'corner': 'bottom_left', 'size_px': 25,
         'opacity': .78, 'margin_px': 34},
        {'url': LOGO_URLS['bottom_right'], 'corner': 'bottom_right', 'size_px': 68,
         'opacity': .72, 'margin_px': 34},
    ],
    'edge_text': [
        {'text': 'STORIES IN MOTION', 'mode': 'static', 'edge': 'top',
         'font_size_px': 20, 'color': '#F5F8FF', 'margin_px': 72},
        {'text': 'IDEAS IN MOTION · THE NEXT STORY STARTS HERE',
         'mode': 'path', 'edges': ['left', 'top', 'right'],
         'font_size_px': 20, 'margin_px': 100,
         'speed_px_per_second': 170, 'color': '#E0FBFF'},
        {'text': 'WATCH · CREATE · SHARE', 'mode': 'marquee',
         'edge': 'bottom', 'direction': 'reverse',
         'font_size_px': 18, 'margin_px': 100,
         'speed_px_per_second': 90, 'color': '#FFEFE6'},
        {'text': 'FILMED FOR THE MOMENT', 'mode': 'marquee',
         'edge': 'right', 'font_size_px': 18, 'margin_px': 112,
         'speed_px_per_second': 95, 'color': '#C8FFF9'},
    ],
}

VIDEO_REQUEST = {
    'source_path': str(SYNCED),
    'request_id': 'kaggle-branding-' + uuid.uuid4().hex,
    'start_seconds': SOURCE_START_SECONDS, 'end_seconds': SOURCE_END_SECONDS,
    'target_seconds': TARGET_SECONDS, 'output_count': 1,
    'preset': 'viral_punch', 'captions_enabled': True,
    'caption_overrides': {
        'uppercase': True, 'animation': 'bounce', 'animation_scope': 'phrase',
        'grouping': 'phrase', 'highlight_mode': 'word', 'max_words': 5,
        'max_chars': 34, 'primary': '&H00FFFFFF', 'highlight': '&H00FFFF00',
    },
    'smart_cut': True, 'remove_silence': True, 'remove_fillers': False,
    'silence_gap_seconds': .62, 'highlight_mode': 'chronological',
    'max_shot_seconds': 12, 'scene_detection': False, 'hook_text': 'off',
    'fit_mode': 'crop', 'crop_x_px': 980, 'crop_y_px': 0, 'crop_scale': 1.0,
    'reframe': 'center', 'video_effect': 'punchy',
    'dynamic_zoom': 'punchy', 'zoom_amount': .12, 'zoom_ramp_seconds': 4,
    'zoom_shots': [1, 3, 5],
    'zoom_overrides': [
        {'shot': 2, 'direction': 'out', 'amount': .16, 'ramp_seconds': 2},
        {'shot': 3, 'direction': 'off'},
    ],
    'transition': 'none', 'transition_sfx': 'none',
    'branding': BRANDING, 'unlimited_resources': True,
}
validated = AutoEditRequest.model_validate(VIDEO_REQUEST)
assert validated.captions_enabled and validated.branding is not None
assert len(validated.branding.gradients) == 4 and len(validated.branding.logos) == 4
assert 0 <= SOURCE_START_SECONDS < SOURCE_END_SECONDS <= 180
print('Validated request:', SOURCE_START_SECONDS, 'to', SOURCE_END_SECONDS,
      '| target', TARGET_SECONDS, 'seconds |', len(BRANDING['logos']), 'sample logos')


## 5. Start SH01, check its endpoints and sample image URLs

The HTTP server stays alive across the remaining cells. If port 5000 is occupied, stop the previous server or restart Kaggle. This cell also checks the Cloudflare queue: the job cannot run if that shared store is unavailable.

In [ ]:
import time
import requests

BASE = 'http://127.0.0.1:5000'
HEADERS = {'X-API-KEY': os.environ['MEDIA_CLIP_API_KEY']}
SERVER_LOG = INPUT_DIR / 'branding-test-server.log'

try:
    if requests.get(BASE + '/openapi.json', timeout=2).ok:
        raise RuntimeError('Another process already uses port 5000; stop it before continuing')
except requests.exceptions.ConnectionError:
    pass

q = requests.get(os.environ['CONTROL_PLANE_BASE_URL'] + '/internal/media-jobs/queue/health',
                 headers={'X-MEDIA-JOB-KEY': os.environ['MEDIA_JOB_STORE_API_KEY']}, timeout=20)
q.raise_for_status()
print('Shared queue:', q.json())

for label, url in {**LOGO_URLS, 'image_background': IMAGE_BACKGROUND_URL}.items():
    response = requests.get(url, stream=True, timeout=25)
    try:
        response.raise_for_status()
        content_type = response.headers.get('Content-Type', '').lower()
        if 'image' not in content_type:
            raise RuntimeError(f'{label} is not an image: {content_type} ({url})')
        print('Reachable image:', label, content_type.split(';')[0])
    finally:
        response.close()

server_log_stream = SERVER_LOG.open('w')
server_proc = subprocess.Popen([
    sys.executable, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1',
    '--port', '5000', '--workers', '1',
], cwd=REPO, env=os.environ.copy(), stdin=subprocess.DEVNULL,
   stdout=server_log_stream, stderr=subprocess.STDOUT)
try:
    for _ in range(90):
        if server_proc.poll() is not None:
            raise RuntimeError('SH01 exited during startup:\n' + SERVER_LOG.read_text()[-5000:])
        try:
            if requests.get(BASE + '/openapi.json', timeout=2).ok:
                break
        except requests.exceptions.RequestException:
            pass
        time.sleep(2)
    else:
        raise TimeoutError('SH01 did not become ready:\n' + SERVER_LOG.read_text()[-5000:])
    capabilities = requests.get(BASE + '/media/auto-edit-capabilities', headers=HEADERS, timeout=20)
    capabilities.raise_for_status()
    cap = capabilities.json()
    assert cap['unlimited_resources'] is True and cap['render_size'] == [1080, 1920]
    presets = requests.get(BASE + '/media/auto-edit-presets', headers=HEADERS, timeout=20)
    presets.raise_for_status()
    print('Server ready:', BASE, '| render:', cap['render_size'])
    print('Preset names:', ', '.join(presets.json()['presets']))
except Exception:
    server_proc.terminate()
    server_proc.wait(timeout=15)
    raise


## 6. Test the image-template endpoint (gradients, logos, static edge labels)

The image route intentionally rejects animation: images show only the **static** edge text. The public GitHub icon used as a background is a functional placeholder; set `IMAGE_BACKGROUND_URL` for an attractive photograph. The response is saved and displayed.

In [ ]:
from IPython.display import Image as NotebookImage, display

IMAGE_BRANDING = {**BRANDING, 'edge_text': [
    {'text': 'TOP EDGE', 'mode': 'static', 'edge': 'top',
     'font_size_px': 18, 'color': '#FFFFFF', 'margin_px': 110},
    {'text': 'RIGHT EDGE', 'mode': 'static', 'edge': 'right',
     'font_size_px': 18, 'color': '#CCFFFF', 'margin_px': 106},
    {'text': 'BOTTOM EDGE', 'mode': 'static', 'edge': 'bottom',
     'font_size_px': 18, 'color': '#FFFFFF', 'margin_px': 110},
    {'text': 'LEFT EDGE', 'mode': 'static', 'edge': 'left',
     'font_size_px': 18, 'color': '#FFD9CC', 'margin_px': 106},
]}
IMAGE_REQUEST = {
    'background_url': IMAGE_BACKGROUND_URL,
    'text': 'MAKE EVERY SECOND COUNT',
    'text_color': '#FFFFFF', 'text_font_size_px': 62,
    'style_preset': 'viral_punch', 'branding': IMAGE_BRANDING,
    'fmt': 'png', 'upload_to_imagekit': False,
}
RenderTemplateRequest.model_validate(IMAGE_REQUEST)
img_response = requests.post(BASE + '/img/render', json=IMAGE_REQUEST, timeout=180)
if not img_response.ok:
    raise RuntimeError('Image endpoint rejected test: ' + img_response.text[:1700])
assert img_response.headers.get('Content-Type', '').startswith('image/png')
IMAGE_OUTPUT = INPUT_DIR / 'branding-image-template-test.png'
IMAGE_OUTPUT.write_bytes(img_response.content)
print('Saved image endpoint result:', IMAGE_OUTPUT, '| bytes:', IMAGE_OUTPUT.stat().st_size)
display(NotebookImage(filename=str(IMAGE_OUTPUT), width=360))


## 7. Submit the video job and record stage timings

The request reaches `POST /media/auto-edit`, and SH01 does the transcription, cuts, captions, zoom, video branding, B2 upload, and final encoding. The notebook only polls the API and downloads the completed video. Stages are printed as they change and stored in `branding-test-timings.json`. The `job_id` is saved as soon as SH01 accepts it.

In [ ]:
from datetime import datetime, timezone

VIDEO_OUTPUT = INPUT_DIR / 'final-branding-endpoint-test.mp4'
STATE_FILE = INPUT_DIR / 'branding-test-job-state.json'
JOB_ID_FILE = INPUT_DIR / 'branding-test-job-id.txt'
REQUEST_FILE = INPUT_DIR / 'branding-test-request.json'
TIMING_FILE = INPUT_DIR / 'branding-test-timings.json'
REQUEST_FILE.write_text(json.dumps(VIDEO_REQUEST, indent=2))

submit = requests.post(BASE + '/media/auto-edit', headers=HEADERS,
                       json=VIDEO_REQUEST, timeout=90)
if not submit.ok:
    raise RuntimeError('Submission failed: ' + submit.text[:1700])
accepted = submit.json()
job_id = accepted['job_id']
JOB_ID_FILE.write_text(job_id)
print('Accepted:', job_id, '| status:', accepted.get('status'), flush=True)

started = time.monotonic()
stage_started = started
last_stage = None
stage_times = []
last_update = 0
job = accepted
for attempt in range(360):  # Up to one hour; resume using the saved job ID if needed.
    time.sleep(10)
    try:
        response = requests.get(BASE + f'/media/auto-edit-jobs/{job_id}',
                                headers=HEADERS, timeout=30)
        response.raise_for_status()
    except requests.exceptions.RequestException as exc:
        print('Temporary status failure:', type(exc).__name__, flush=True)
        continue
    job = response.json()
    STATE_FILE.write_text(json.dumps(job, indent=2))
    stage = job.get('stage', job['status'])
    now = time.monotonic()
    if stage != last_stage:
        if last_stage is not None:
            stage_times.append({'stage': last_stage,
                                'approx_seconds': round(now - stage_started, 1)})
        stage_started = now
        last_stage = stage
    if stage != last_update or (attempt + 1) % 6 == 0:
        print(datetime.now(timezone.utc).strftime('%H:%M:%SZ'), '|',
              job['status'], '|', stage, '| elapsed', round(now - started), 's', flush=True)
        last_update = stage
    TIMING_FILE.write_text(json.dumps({'job_id': job_id, 'revision': revision,
                                       'elapsed_seconds': round(now - started, 1),
                                       'completed_stages': stage_times,
                                       'current_stage': stage}, indent=2))
    if job['status'] in ('complete', 'failed', 'interrupted'):
        stage_times.append({'stage': stage, 'approx_seconds': round(now - stage_started, 1)})
        break
else:
    raise TimeoutError('Still running. Job ID: ' + job_id + '; state saved at ' + str(STATE_FILE))

if job['status'] != 'complete':
    print('Failure details:', job.get('failed_stage'), job.get('error'))
    print('Recent server log:\n' + SERVER_LOG.read_text()[-4000:])
    raise RuntimeError('SH01 did not complete job ' + job_id)

output = job['outputs'][0]
token = output['token']
download = requests.get(BASE + f'/media/rendered/{token}', timeout=900, stream=True)
download.raise_for_status()
partial = VIDEO_OUTPUT.with_suffix('.part')
with partial.open('wb') as f:
    for chunk in download.iter_content(1024 * 1024):
        if chunk:
            f.write(chunk)
download.close()
partial.replace(VIDEO_OUTPUT)
if not VIDEO_OUTPUT.stat().st_size:
    raise RuntimeError('Downloaded video is empty')
print('Saved video:', VIDEO_OUTPUT, '| bytes:', VIDEO_OUTPUT.stat().st_size)
print('Stage times:', stage_times)


## 8. Check what SH01 actually selected and rendered

The edit decision list proves which **shot numbers** received zoom and how many source intervals were skipped. Stream metadata verifies the 1080×1920 video and audio start times. These checks cannot prove whether the edit looks good, whether a face is framed well, or whether word highlighting is visually perfect: download and watch the MP4.

In [ ]:
import collections
from IPython.display import FileLink

job = json.loads(STATE_FILE.read_text())
assert job['status'] == 'complete' and VIDEO_OUTPUT.is_file()
rendered = job['outputs'][0]
shots = rendered['edit_decision_list']
assert shots, 'No selected shots reported'
stream_info = probe(VIDEO_OUTPUT)
video = next(s for s in stream_info['streams'] if s['codec_type'] == 'video')
audio = next(s for s in stream_info['streams'] if s['codec_type'] == 'audio')
assert (video['width'], video['height']) == (1080, 1920)
assert abs(float(video.get('start_time', 0))) < .12
assert abs(float(audio.get('start_time', 0))) < .12

print('SH01 revision:', revision, '| job:', job['job_id'])
print('Rendered duration:', rendered['duration'], 's', '| words:', rendered.get('word_count'))
print('Requested source window:', (SOURCE_START_SECONDS, SOURCE_END_SECONDS))
print('Candidates:', job.get('analysis', {}).get('candidate_segments'),
      '| selected shots:', len(shots))
print('Shot reasons:', dict(collections.Counter(s.get('reason', '?') for s in shots)))
print('Shot | source (seconds) | output (seconds) | zoom direction | amount | ramp (seconds)')
for n, shot in enumerate(shots, 1):
    assert SOURCE_START_SECONDS - .1 <= float(shot['source_start'])
    assert float(shot['source_end']) <= SOURCE_END_SECONDS + .1
    zoom = shot.get('zoom') or {}
    print(f"{n:>4} | {shot['source_start']:>6}–{shot['source_end']:<6} | "
          f"{shot['output_start']:>6} | {zoom.get('direction'):>3} | "
          f"{zoom.get('amount')} | {zoom.get('ramp_seconds')}")
assert shots[0]['zoom']['direction'] != 'off', 'Shot 1 should zoom in'
if len(shots) >= 2:
    assert shots[1]['zoom']['direction'] == 'out', 'Shot 2 override was not applied'
if len(shots) >= 3:
    assert shots[2]['zoom']['direction'] == 'off', 'Shot 3 off override was not applied'

removed = []
previous_end = SOURCE_START_SECONDS
for n, shot in enumerate(shots, 1):
    gap = float(shot['source_start']) - previous_end
    if gap > .1:
        removed.append((n, round(gap, 2)))
    previous_end = float(shot['source_end'])
print('Source gaps before selected shots:', removed[:20])
print('Total selected source time:', round(sum(float(s['duration']) for s in shots), 2), 's')
print('Audio/video start:', audio.get('start_time'), '/', video.get('start_time'))
display(FileLink(str(VIDEO_OUTPUT)))


In [ ]:
from IPython.display import Video, display

display(Video(
    VIDEO_OUTPUT,
    embed=True
))

## 9. Optional caption-off validation and shutdown

`captions_enabled=false` prevents adding captions to an already-captioned source. You can validate the request without rerendering. If you want a second rendered output for visual comparison, create a new request ID, change that flag to false, and repeat cells 7–8. Stop the Kaggle server after downloading and inspecting your output.

In [ ]:
captionless = dict(VIDEO_REQUEST, captions_enabled=False,
                   request_id='captionless-check-' + uuid.uuid4().hex)
validated_captionless = AutoEditRequest.model_validate(captionless)
assert validated_captionless.captions_enabled is False
print('Caption-off request validates; no second render was submitted.')

# Run this final line only once the rendered output has been downloaded.
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    server_proc.wait(timeout=20)
    print('Kaggle SH01 server stopped. Outputs remain in', INPUT_DIR)
